# Pipecat Cascade + LangSmith

This notebook keeps the workshop-facing pieces visible: the agent setup, the LangSmith tracing setup, and the final voice loop. Local mic/speaker transport, audio recording, and runner cleanup live in `voice_demo.workshop`.

## 1. Agent Setup

The cascade has three model-facing stages: speech-to-text, a LangGraph-backed LLM service, and text-to-speech. The graph owns tool use and reasoning; Pipecat owns streaming audio between stages.

In [ ]:
import os
import uuid

from dotenv import load_dotenv
from pipecat.services.openai.stt import OpenAISTTService
from pipecat.services.openai.tts import OpenAITTSService

from voice_demo.pipecat_with_langgraph.graph import GREETING, SYSTEM_PROMPT, build_graph
from voice_demo.pipecat_with_langgraph.langgraph_llm_service import LangGraphLLMService
from voice_demo.workshop import cascading_pipeline, run_task

load_dotenv()

PROJECT = "voice-workshop-pipecat-cascade"
STT_MODEL = os.getenv("PIPECAT_STT_MODEL", "gpt-4o-mini-transcribe")
LLM_MODEL = os.getenv("PIPECAT_LLM_MODEL", "gpt-4o-mini")
TTS_VOICE = os.getenv("PIPECAT_TTS_VOICE", "alloy")

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook."
assert os.getenv("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY before running this notebook."

In [ ]:
stt = OpenAISTTService(settings=OpenAISTTService.Settings(model=STT_MODEL))

llm = LangGraphLLMService(
    graph=build_graph(SYSTEM_PROMPT),
    settings=LangGraphLLMService.Settings(
        model=LLM_MODEL,
        system_instruction=SYSTEM_PROMPT,
    ),
)

tts = OpenAITTSService(settings=OpenAITTSService.Settings(voice=TTS_VOICE))

## 2. Setting Up Tracing

The Pipecat integration reads LangSmith settings from the environment. For the workshop, tracing setup is just a fresh thread id plus `configure_pipecat`.

In [ ]:
from langsmith.integrations.pipecat import configure_pipecat, set_thread_id

conversation_id = str(uuid.uuid4())
set_thread_id(conversation_id)

span_processor = configure_pipecat(
    project=PROJECT,
    service_name="workshop-cascade",
    llm_span_kind="chain",
)
assert span_processor is not None

## 3. Running the Voice Agent

The helper below assembles the Pipecat pipeline, places the recorder after speaker output so LangSmith gets what the user actually heard, and runs until you stop the cell.

In [ ]:
from pipecat.frames.frames import TTSSpeakFrame

pipeline, audiobuffer = cascading_pipeline(
    stt=stt,
    llm=llm,
    tts=tts,
    span_processor=span_processor,
    conversation_id=conversation_id,
)

await audiobuffer.start_recording()
await run_task(
    pipeline,
    conversation_id,
    before_run=[TTSSpeakFrame(text=GREETING, append_to_context=True)],
)